In [25]:
import pandas as pd
import os
import numpy as np
from datetime import datetime, timedelta


# Caminhos de entrada e saída
json_path = "noticias_html.json"
json_path_2 =  "noticias.json"
output_dir = "silver-json-to_parquet/noticias_conteudo_parquet"

# Criar pasta de saída se não existir
os.makedirs(output_dir, exist_ok=True)
print("Lendo Json")

df1 = pd.read_json(json_path)

#Fazendo join com tabela de noticias para trazer e particionar pela data de processamento tbm, como se fosse uma data de hub
#era pra ter adicionado isso na bronze mas esqueci rsrsrs entáo fiz a extracao da silver da de noticias pq ja ta com o campo de processament criado
#como se essa fosse uma `tabela dependente` por seu conteudo principal ser mt grande e já ser
#um dado tratato json

inicio_ano = 2024
fim = 2025
df_parquet = pd.DataFrame()

while inicio_ano <= fim:
    ano = str(inicio_ano)
    print(f"Lendo ano de {ano}")
    mes = 1
    fim_mes = 12
    while mes <= fim_mes:
        
        print(f"Lendo {mes} mes")
        try:
            df_parquet_temp = pd.read_parquet(f"silver-json-to_parquet/noticias_parquet/ano={ano}/mes={mes}/dados_noticias_{ano}_{mes}.parquet")
            df_parquet = pd.concat([df_parquet, df_parquet_temp], ignore_index=True)
            print(f"Leitura {ano}/{mes} concluida")
            mes = mes + 1
        except Exception as e:
            # trata o erro
            print("Ocorreu um erro:", e)
        finally:
            mes = mes + 1
    inicio_ano = inicio_ano + 1
            

df2 = df_parquet[['ano', 'mes', 'data_processamento', 'link']]

df = pd.merge(
    df1, 
    df2, 
    left_on='id_pesquisa',   
    right_on='link',        
    how='inner'     
)

df = df.drop_duplicates()
total_parquet = 0
anos_processados = []

print("Gerando arquivos Parquet por ano:")
for ano, grupo in df.groupby('ano'):
    for mes, grupo in df.groupby('mes'):
        path_ano = os.path.join(output_dir, f"ano={ano}", f"mes={mes}" )
        os.makedirs(path_ano, exist_ok=True)

        parquet_path = os.path.join(path_ano, f"dados_conteudo_noticias_{ano}_{mes}.parquet")
        grupo = grupo.astype({
            "ano": "int64",
            "mes": "int64",
            "id_pesquisa": "string",
            "html": "string",
            "data_processamento": "string"        })
        grupo.to_parquet(
                parquet_path, engine='pyarrow', index=False,
                use_dictionary={"ano": False, "mes": False},
                coerce_timestamps="ms",
                allow_truncated_timestamps=True,
            )

        qtd = len(grupo)
        total_parquet += qtd
        anos_processados.append((ano, qtd))
        print(f" Ano/Mesref: {ano}/{mes}: {qtd:,} registros → {parquet_path}")



Lendo Json
Lendo ano de 2024
Lendo 1 mes
Leitura 2024/1 concluida
Lendo 3 mes
Leitura 2024/3 concluida
Lendo 5 mes
Leitura 2024/5 concluida
Lendo 7 mes
Leitura 2024/7 concluida
Lendo 9 mes
Leitura 2024/9 concluida
Lendo 11 mes
Leitura 2024/11 concluida
Lendo ano de 2025
Lendo 1 mes
Leitura 2025/1 concluida
Lendo 3 mes
Leitura 2025/3 concluida
Lendo 5 mes
Leitura 2025/5 concluida
Lendo 7 mes
Leitura 2025/7 concluida
Lendo 9 mes
Leitura 2025/9 concluida
Lendo 11 mes
Leitura 2025/11 concluida
Gerando arquivos Parquet por ano:
 Ano/Mesref: 2024/1: 13 registros → silver-json-to_parquet/noticias_conteudo_parquet/ano=2024/mes=1/dados_conteudo_noticias_2024_1.parquet
 Ano/Mesref: 2024/3: 28 registros → silver-json-to_parquet/noticias_conteudo_parquet/ano=2024/mes=3/dados_conteudo_noticias_2024_3.parquet
 Ano/Mesref: 2024/5: 11 registros → silver-json-to_parquet/noticias_conteudo_parquet/ano=2024/mes=5/dados_conteudo_noticias_2024_5.parquet
 Ano/Mesref: 2024/7: 26 registros → silver-json-to_par

<h2> Tabela ja particionada pela mesma data da principal para facilitar joins e dificultar full load na camada ouro

In [26]:
df

,id_pesquisa,html,ano,mes,data_processamento,link
0,https://www.serasa.com.br/limpa-nome-online/bl...,endividamento do brasileiro - Pesquisa Google ...,2025,3,2025-03-13,https://www.serasa.com.br/limpa-nome-online/bl...
2,https://www.cnnbrasil.com.br/economia/macroeco...,endividamento do brasileiro - Pesquisa Google ...,2025,1,2025-01-27,https://www.cnnbrasil.com.br/economia/macroeco...
4,https://www.gazetadopovo.com.br/economia/crise...,endividamento do brasileiro - Pesquisa Google ...,2024,11,2024-11-01,https://www.gazetadopovo.com.br/economia/crise...
6,https://www.poder360.com.br/poder-economia/end...,endividamento do brasileiro - Pesquisa Google ...,2024,11,2024-11-23,https://www.poder360.com.br/poder-economia/end...
8,https://www.gazetadopovo.com.br/economia/inadi...,endividamento do brasileiro - Pesquisa Google ...,2025,7,2025-07-27,https://www.gazetadopovo.com.br/economia/inadi...
...,...,...,...,...,...,...
230,https://macae-airport.com/,voos internacionais brasil noticias - Pesquisa...,2025,9,2025-09-09,https://macae-airport.com/
232,https://www.melhoresdestinos.com.br/passagens-...,voos internacionais brasil noticias - Pesquisa...,2025,5,2025-05-02,https://www.melhoresdestinos.com.br/passagens-...
234,https://embratur.com.br/2025/10/08/com-quase-1...,voos internacionais brasil noticias - Pesquisa...,2025,3,2025-03-02,https://embratur.com.br/2025/10/08/com-quase-1...
236,https://www.travelandtourworld.com.br/not%C3%A...,voos internacionais brasil noticias - Pesquisa...,2025,1,2025-01-22,https://www.travelandtourworld.com.br/not%C3%A...


<H2> Fazendo verificacao de campos nulos

In [27]:
df.eq('').sum()

id_pesquisa           0
html                  0
ano                   0
mes                   0
data_processamento    0
link                  0
dtype: Int64

<H2> Fazendo verificacao de duplicados

In [28]:
len(df2), len(df)

(208, 120)